# Загрузка данных для задания по рыночному риску (пункт 1)

Тетрадка скачивает **все** данные из пункта 1 задания напрямую из первоисточников,
разрешённых в задании, и сохраняет их в `project_2/data/` в формате **parquet**.

**Источники (только из задания):**

| Данные | Источник | Где именно |
|---|---|---|
| 1.a Кривая бескупонной доходности (КБД) | **ЦБ РФ**, `cbr.ru` | `hd_base/zcyc_params` |
| 1.b Описания и расписания выплат ОФЗ | **MOEX**, `moex.ru` | ISS `/securities/{secid}` и `/bondization` |
| 1.c Котировки облигаций | **MOEX** | ISS history, рынок `bonds`, доска `TQOB` |
| 1.d Котировки акций | **MOEX** | ISS history, рынок `shares`, доска `TQBR` |
| 1.e Индексы МосБиржи и РТС | **MOEX** | ISS history, рынок `index` |
| 1.e Нефть Brent | **MOEX** | ISS history, FORTS, фьючерсы `BR` (фронт-месяц) |
| 1.e Курсы USD, EUR | **ЦБ РФ** | сервис `XML_dynamic` |
| 1.f Фьючерс + опционы | **MOEX** | ISS FORTS, рынки `forts` и `options` |

Вся бизнес-логика вынесена в `project_2/utils/` — тетрадка лишь оркестрирует вызовы,
чтобы эксперимент можно было воспроизвести целиком.

In [2]:
# --- Настройка окружения ---
import sys
from pathlib import Path

# Делаем папку project_2 видимой для импорта utils (работает и из тетрадки, и из nbconvert)
PROJ = Path.cwd()
if not (PROJ / "utils").exists() and (PROJ.parent / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))

import numpy as np
import pandas as pd

from src import config as C
from src.data_collection import sources as S

np.random.seed(C.RANDOM_SEED)  # фиксируем seed (требование воспроизводимости)

session = S._session()          # один HTTP-сеанс на всю загрузку
DATA = C.DATA_DIR
print("Период:", C.START_DATE, "—", C.END_DATE)
print("Каталог данных:", DATA)

Период: 2021-01-01 — 2026-01-01
Каталог данных: C:\Users\Алексей\Desktop\project_2\data


In [2]:
# --- Помощник сохранения в parquet с краткой сводкой ---
def save_parquet(df: pd.DataFrame, name: str) -> Path:
    path = DATA / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"  [OK] {name}.parquet  ->  строк: {len(df):>6}, колонок: {df.shape[1]}")
    return path

## 1.a Кривая бескупонной доходности (КБД), ЦБ РФ

Безрисковые спот-доходности гособлигаций по срокам **0.25 … 30 лет** (% годовых).
ЦБ публикует уже готовую таблицу значений (методология Нельсона—Сигеля—Свенссона по
сделкам с ОФЗ на MOEX). Качаем по годам и склеиваем.

In [3]:
zcyc = S.get_cbr_zcyc(session, C.START_DATE, C.END_DATE)
print("Период КБД:", zcyc['DATE'].min().date(), "—", zcyc['DATE'].max().date())
print("Сроки (лет):", [c for c in zcyc.columns if c != 'DATE'])
save_parquet(zcyc, "zcyc_cbr")
zcyc.tail(3)

Период КБД: 2021-01-04 — 2025-12-30
Сроки (лет): ['0,25', '0,5', '0,75', '1', '2', '3', '5', '7', '10', '15', '20', '30']
  [OK] zcyc_cbr.parquet  ->  строк:   1256, колонок: 13


,DATE,"0,25","0,5","0,75",1,2,3,5,7,10,15,20,30
1253,2025-12-26,12.72,13.09,13.40,13.66,14.34,14.66,14.80,14.70,14.44,14.07,13.85,13.68
1254,2025-12-29,12.72,13.06,13.35,13.60,14.25,14.55,14.72,14.65,14.44,14.12,13.93,13.76
1255,2025-12-30,12.10,12.50,12.84,13.14,13.92,14.30,14.58,14.59,14.44,14.11,13.91,13.79


## 1.b Описания ОФЗ и расписания выплат, MOEX (5 нужно — берём 10 с запасом)

Критерии задания: гособлигации с полностью известными выплатами (фиксированный купон,
**не** привязанный к показателям), **без оферт**, с погашением **после 01.01.2026**.
Этим критериям отвечают выпуски **ОФЗ-ПД**. Резолвим полные SECID, выгружаем карточки,
проверяем критерии, сохраняем расписания купонов.

In [4]:
# Резолвим короткие номера выпусков в полные SECID и собираем карточки
desc_rows, coupon_frames, secids = [], [], {}
for num in C.OFZ_PD_NUMBERS:
    secid = S.resolve_ofz_secid(session, num)
    secids[num] = secid
    d = S.get_bond_description(session, secid)
    desc_rows.append({
        "NUMBER": num, "SECID": secid, "ISIN": d.get("ISIN"),
        "NAME": d.get("NAME"), "MATDATE": d.get("MATDATE"),
        "COUPONPERCENT": d.get("COUPONPERCENT"), "COUPONPERIOD": d.get("COUPONPERIOD"),
        "FACEVALUE": d.get("FACEVALUE"), "TYPE": d.get("TYPE"),
        "OFFERDATE": d.get("OFFERDATE"),       # должно быть пусто => без оферты
    })
    cp, am, of = S.get_bond_schedule(session, secid)
    cp = cp.copy(); cp["NUMBER"] = num
    coupon_frames.append(cp)

bonds_desc = pd.DataFrame(desc_rows)
bonds_desc["MATDATE"] = pd.to_datetime(bonds_desc["MATDATE"])
# Проверка критериев задания:
bonds_desc["after_2026"] = bonds_desc["MATDATE"] > pd.Timestamp("2026-01-01")
bonds_desc["no_offer"]   = bonds_desc["OFFERDATE"].isna()
assert bonds_desc["after_2026"].all(), "Есть выпуск с погашением до 2026!"
assert bonds_desc["no_offer"].all(),   "Есть выпуск с офертой!"
save_parquet(bonds_desc, "bonds_descriptions")
bonds_desc[["NUMBER","SECID","NAME","MATDATE","COUPONPERCENT","OFFERDATE","after_2026","no_offer"]]

  [OK] bonds_descriptions.parquet  ->  строк:     10, колонок: 12


,NUMBER,SECID,NAME,MATDATE,COUPONPERCENT,OFFERDATE,after_2026,no_offer
0,26207,SU26207RMFS9,ОФЗ-ПД 26207 03/02/27,2027-02-03,8.15,None,True,True
1,26212,SU26212RMFS9,ОФЗ-ПД 26212 19/01/28,2028-01-19,7.05,None,True,True
2,26219,SU26219RMFS4,ОФЗ-ПД 26219 16/09/26,2026-09-16,7.75,None,True,True
3,26226,SU26226RMFS9,ОФЗ-ПД 26226 07/10/26,2026-10-07,7.95,None,True,True
4,26224,SU26224RMFS4,ОФЗ-ПД 26224 23/05/29,2029-05-23,6.9,None,True,True
5,26228,SU26228RMFS5,ОФЗ-ПД 26228 10/04/30,2030-04-10,7.65,None,True,True
6,26218,SU26218RMFS6,ОФЗ-ПД 26218 17/09/31,2031-09-17,8.5,None,True,True
7,26221,SU26221RMFS0,ОФЗ-ПД 26221 23/03/33,2033-03-23,7.7,None,True,True
8,26225,SU26225RMFS1,ОФЗ-ПД 26225 10/05/34,2034-05-10,7.25,None,True,True
9,26230,SU26230RMFS1,ОФЗ-ПД 26230 16/03/39,2039-03-16,7.7,None,True,True


In [5]:
# Расписания купонов всех выбранных ОФЗ (полностью известные размеры выплат)
bonds_coupons = pd.concat(coupon_frames, ignore_index=True)
bonds_coupons["coupondate"] = pd.to_datetime(bonds_coupons["coupondate"])
save_parquet(bonds_coupons, "bonds_coupons")
bonds_coupons[["NUMBER","secid","coupondate","value","valueprc","facevalue"]].head(5)

  [OK] bonds_coupons.parquet  ->  строк:    279, колонок: 15


,NUMBER,secid,coupondate,value,valueprc,facevalue
0,26207,SU26207RMFS9,2012-08-22,40.64,8.15,1000
1,26207,SU26207RMFS9,2013-02-20,40.64,8.15,1000
2,26207,SU26207RMFS9,2013-08-21,40.64,8.15,1000
3,26207,SU26207RMFS9,2014-02-19,40.64,8.15,1000
4,26207,SU26207RMFS9,2014-08-20,40.64,8.15,1000


## 1.c Рыночные котировки выбранных ОФЗ, MOEX (доска TQOB)

Чистая цена `CLOSE` (% от номинала), `LEGALCLOSEPRICE`, накопленный купонный доход
`ACCINT`, доходность `YIELDCLOSE`, дюрация `DURATION`. «Грязная» цена для оценки
портфеля = чистая + НКД.

In [6]:
bh_frames = []
for num, secid in secids.items():
    h = S.get_bond_history(session, secid, C.START_DATE, C.END_DATE)
    h["NUMBER"] = num
    bh_frames.append(h)
bonds_history = pd.concat(bh_frames, ignore_index=True)
bonds_history["TRADEDATE"] = pd.to_datetime(bonds_history["TRADEDATE"])
for c in ["CLOSE","LEGALCLOSEPRICE","ACCINT","WAPRICE","YIELDCLOSE","DURATION","VOLUME","VALUE"]:
    bonds_history[c] = pd.to_numeric(bonds_history[c], errors="coerce")
print("Диапазон дат:", bonds_history['TRADEDATE'].min().date(), "—", bonds_history['TRADEDATE'].max().date())
print("Бумаг:", bonds_history['SECID'].nunique())
save_parquet(bonds_history, "bonds_history")
bonds_history.groupby('NUMBER')['TRADEDATE'].agg(['min','max','count'])

Диапазон дат: 2021-01-04 — 2025-12-30
Бумаг: 10
  [OK] bonds_history.parquet  ->  строк:  12710, колонок: 15


,min,max,count
NUMBER,,,
26207,2021-01-04,2025-12-30,1271
26212,2021-01-04,2025-12-30,1271
26218,2021-01-04,2025-12-30,1271
26219,2021-01-04,2025-12-30,1271
26221,2021-01-04,2025-12-30,1271
26224,2021-01-04,2025-12-30,1271
26225,2021-01-04,2025-12-30,1271
26226,2021-01-04,2025-12-30,1271
26228,2021-01-04,2025-12-30,1271


## 1.d Котировки российских акций, MOEX (доска TQBR; 10 нужно — берём 12)

In [7]:
sh_frames = []
for t in C.STOCK_TICKERS:
    h = S.get_share_history(session, t, C.START_DATE, C.END_DATE)
    sh_frames.append(h)
stocks_history = pd.concat(sh_frames, ignore_index=True)
stocks_history["TRADEDATE"] = pd.to_datetime(stocks_history["TRADEDATE"])
for c in ["CLOSE","LEGALCLOSEPRICE","WAPRICE","OPEN","HIGH","LOW","VOLUME","VALUE"]:
    stocks_history[c] = pd.to_numeric(stocks_history[c], errors="coerce")
print("Тикеры:", sorted(stocks_history['SECID'].unique()))
save_parquet(stocks_history, "stocks_history")
stocks_history.groupby('SECID')['TRADEDATE'].agg(['min','max','count'])

Тикеры: ['CHMF', 'GAZP', 'GMKN', 'LKOH', 'MGNT', 'MTSS', 'NVTK', 'PLZL', 'ROSN', 'SBER', 'SNGS', 'TATN']
  [OK] stocks_history.parquet  ->  строк:  15252, колонок: 11


,min,max,count
SECID,,,
CHMF,2021-01-04,2025-12-30,1271
GAZP,2021-01-04,2025-12-30,1271
GMKN,2021-01-04,2025-12-30,1271
LKOH,2021-01-04,2025-12-30,1271
MGNT,2021-01-04,2025-12-30,1271
MTSS,2021-01-04,2025-12-30,1271
NVTK,2021-01-04,2025-12-30,1271
PLZL,2021-01-04,2025-12-30,1271
ROSN,2021-01-04,2025-12-30,1271


## 1.e Индексы МосБиржи (IMOEX) и РТС (RTSI), MOEX

In [8]:
idx_frames = []
for t in C.INDEX_TICKERS:
    h = S.get_index_history(session, t, C.START_DATE, C.END_DATE)
    idx_frames.append(h)
indices_history = pd.concat(idx_frames, ignore_index=True)
indices_history["TRADEDATE"] = pd.to_datetime(indices_history["TRADEDATE"])
for c in ["CLOSE","OPEN","HIGH","LOW","VALUE"]:
    indices_history[c] = pd.to_numeric(indices_history[c], errors="coerce")
save_parquet(indices_history, "indices_history")
indices_history.groupby('SECID')['TRADEDATE'].agg(['min','max','count'])

  [OK] indices_history.parquet  ->  строк:   2507, колонок: 7


,min,max,count
SECID,,,
IMOEX,2021-01-04,2025-12-30,1254
RTSI,2021-01-04,2025-12-30,1253


## 1.e Нефть Brent — непрерывный фронт-месяц из фьючерсов BR (FORTS), MOEX

Спот Brent в открытом доступе не публикуется, поэтому строим непрерывный ряд: на каждый
торговый день берём контракт `BR` с **максимальным дневным объёмом** (всегда ближний
ликвидный фьючерс). Цена — в долларах за баррель.
*(Дольше других пунктов: перебираем все месячные контракты.)*

In [9]:
brent = S.get_brent_front_month(session, C.START_DATE, C.END_DATE, C.BRENT_ASSETCODE)
brent["TRADEDATE"] = pd.to_datetime(brent["TRADEDATE"])
print("Диапазон:", brent['TRADEDATE'].min().date(), "—", brent['TRADEDATE'].max().date(), "| дней:", len(brent))
save_parquet(brent, "brent_history")
brent.tail(3)

Диапазон: 2021-01-04 — 2025-12-30 | дней: 1270
  [OK] brent_history.parquet  ->  строк:   1270, колонок: 5


,TRADEDATE,FRONT_CONTRACT,BRENT_USD,VOLUME,OPENPOSITION
1267,2025-12-26,BRF6,61.57,158440.0,153080
1268,2025-12-29,BRF6,62.10,248776.0,127802
1269,2025-12-30,BRF6,62.02,130818.0,98398


## 1.e Официальные курсы USD и EUR, ЦБ РФ

Официальный курс ЦБ за 1 единицу валюты (нормирован на номинал). Непрерывный источник на
всём интервале (биржевые торги USD/EUR на MOEX остановлены с июня 2024 г.).

In [10]:
fx_frames = []
for code, val in C.CBR_CURRENCIES.items():
    f = S.get_cbr_fx(session, val, C.START_DATE, C.END_DATE)
    f["CCY"] = code
    fx_frames.append(f)
fx = pd.concat(fx_frames, ignore_index=True)
print("Диапазон:", fx['DATE'].min().date(), "—", fx['DATE'].max().date())
save_parquet(fx, "fx_cbr")
fx.pivot(index='DATE', columns='CCY', values='RATE').tail(3)

Диапазон: 2021-01-01 — 2025-12-31
  [OK] fx_cbr.parquet  ->  строк:   2474, колонок: 3


CCY,EUR,USD
DATE,,
2025-12-27,91.2066,77.6923
2025-12-30,91.4775,77.4466
2025-12-31,92.0938,78.2267


## 1.f Фьючерс и опционы на фьючерс на один актив, один день 2025 г., MOEX

Актив — **фьючерс на доллар США (Si)** (доллар входит в портфель, опционы Si — самые
ликвидные на MOEX). День — **2025-10-15**. Берём ближайший фьючерс с экспирацией
**более чем на 1 месяц** → квартальный **SiZ5** (экспирация 18.12.2025). Опционы той же
серии (Call и Put).

In [11]:
# Фьючерсы Si за выбранный день + выбор ближайшего с экспирацией >= 1 мес
fut_day = S.get_forts_futures_on_date(session, C.FORTS_ASSETCODE, C.FORTS_TRADE_DAY)
fut_specs = S.get_futures_specs(session, fut_day['SECID'].tolist())
front = S.pick_front_future(fut_specs, C.FORTS_TRADE_DAY, C.FORTS_MIN_DAYS_TO_EXPIRY)
fut_day = fut_day.merge(fut_specs[['SECID','LSTDELDATE']], on='SECID', how='left')
fut_day['IS_CHOSEN_FRONT'] = fut_day['SECID'] == front
print("Выбранный фьючерс (>=1 мес до экспирации):", front,
      "| экспирация:", fut_specs.set_index('SECID').loc[front,'LSTDELDATE'].date())
save_parquet(fut_day, f"forts_futures_{C.FORTS_TRADE_DAY}")
fut_day[['SECID','CLOSE','SETTLEPRICE','VOLUME','OPENPOSITION','LSTDELDATE','IS_CHOSEN_FRONT']]

Выбранный фьючерс (>=1 мес до экспирации): SiZ5 | экспирация: 2025-12-18
  [OK] forts_futures_2025-10-15.parquet  ->  строк:      7, колонок: 21


,SECID,CLOSE,SETTLEPRICE,VOLUME,OPENPOSITION,LSTDELDATE,IS_CHOSEN_FRONT
0,SiH6,82755.0,82766.0,31988.0,434034,2026-03-19,False
1,SiH7,96880.0,96567.0,11.0,390,2027-03-18,False
2,SiM6,85559.0,85559.0,1086.0,14682,2026-06-18,False
3,SiM7,99500.0,99500.0,5.0,232,2027-06-17,False
4,SiU6,88891.0,88773.0,71.0,2364,2026-09-17,False
5,SiZ5,80209.0,80176.0,1005347.0,9011172,2025-12-18,True
6,SiZ6,92665.0,92658.0,40.0,1846,2026-12-17,False


In [12]:
# Опционная цепочка на выбранный фьючерс (Call и Put, все страйки)
import re
opt_day = S.get_forts_options_on_date(session, C.FORTS_ASSETCODE, C.FORTS_TRADE_DAY)
# Предфильтр по суффиксу SECID (квартальная серия декабря) ускоряет выгрузку описаний
cand = [x for x in opt_day['SECID'] if re.search(r'(L5|X5)$', x)]
specs = S.get_option_specs(session, cand)
specs['LSTDELDATE'] = pd.to_datetime(specs['LSTDELDATE'])

# Только опционы на выбранный фьючерс с его же датой экспирации
chosen_exp = fut_specs.set_index('SECID').loc[front, 'LSTDELDATE']
keep = specs[(specs['UNDERLYINGASSET'] == front) & (specs['LSTDELDATE'] == chosen_exp)]
chain = opt_day.merge(keep[['SECID','STRIKE','OPTIONTYPE','UNDERLYINGASSET','LSTDELDATE']],
                      on='SECID', how='inner')
chain['STRIKE'] = pd.to_numeric(chain['STRIKE'], errors='coerce')
chain = chain.sort_values(['OPTIONTYPE','STRIKE']).reset_index(drop=True)
print("Опционов в цепочке:", len(chain), "| страйков:", chain['STRIKE'].nunique(),
      "| типы:", chain['OPTIONTYPE'].value_counts().to_dict())
save_parquet(chain, f"forts_options_chain_{C.FORTS_TRADE_DAY}")
chain[['SECID','STRIKE','OPTIONTYPE','CLOSE','SETTLEPRICE','OPENPOSITION','VOLUME']].head(6)

Опционов в цепочке: 186 | страйков: 93 | типы: {'C': 93, 'P': 93}
  [OK] forts_options_chain_2025-10-15.parquet  ->  строк:    186, колонок: 20


,SECID,STRIKE,OPTIONTYPE,CLOSE,SETTLEPRICE,OPENPOSITION,VOLUME
0,Si64000BL5,64000,C,NaN,16279.0,NaN,NaN
1,Si64500BL5,64500,C,NaN,15785.0,NaN,NaN
2,Si65000BL5,65000,C,NaN,15292.0,NaN,NaN
3,Si65500BL5,65500,C,NaN,14800.0,NaN,NaN
4,Si66000BL5,66000,C,NaN,14309.0,NaN,NaN
5,Si66500BL5,66500,C,NaN,13820.0,NaN,NaN


## Итог: сохранённые файлы

In [13]:
files = sorted(DATA.glob('*.parquet'))
report = []
for f in files:
    df = pd.read_parquet(f)
    report.append({"файл": f.name, "строк": len(df), "колонок": df.shape[1],
                   "размер_КБ": round(f.stat().st_size/1024, 1)})
pd.DataFrame(report)

,файл,строк,колонок,размер_КБ
0,bonds_coupons.parquet,279,15,15.2
1,bonds_descriptions.parquet,10,12,7.1
2,bonds_history.parquet,12710,15,536.3
3,brent_history.parquet,1270,5,36.6
4,forts_futures_2025-10-15.parquet,7,21,12.5
5,forts_options_chain_2025-10-15.parquet,186,20,19.4
6,fx_cbr.parquet,2474,3,37.0
7,indices_history.parquet,2507,7,100.3
8,stocks_history.parquet,15252,11,813.3
9,zcyc_cbr.parquet,1256,13,70.4
